# Airline Satisfaction — CatBoost baseline

This notebook trains a binary classifier for `satisfaction`, evaluates it with **ROC AUC** on a stratified holdout, then fits on all labeled rows and writes `submission_catboost.csv`. It assumes this notebook and the two CSV files are in the same folder.


In [2]:
# Run once if CatBoost is missing from this notebook environment.
%pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [4]:
# Files are next to this notebook. The training CSV was named "train (1).csv".
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = "satisfaction"
ID_COL = "id"

print("train:", train.shape, "test:", test.shape)
display(train.head())
print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False))
print("\nMissing values (training):")
print(train.isna().sum().sort_values(ascending=False).head(10))


train: (699635, 23) test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False



Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Missing values (training):
Arrival Delay in Minutes      292
id                              0
Leg room service                0
Class                           0
Type of Travel                  0
Customer Type                   0
Gender                          0
Departure Delay in Minutes      0
Cleanliness                     0
Checkin service                 0
dtype: int64


In [5]:
# Separate identifiers and target; do not train on the ID by default.
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

# CatBoost accepts categorical columns directly; represent missing categories explicitly.
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns use consistent numeric values and preserve missing values for CatBoost.
for col in X.columns.difference(cat_features):
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical features:", cat_features)
print("Positive class rate:", y.mean())


Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Positive class rate: 0.44357272006117476


In [6]:
# Compare the current Travel_Context features against Travel_Context plus service-rating summaries.
# Both models use exactly the same validation rows and CatBoost settings.
from sklearn.model_selection import train_test_split

row_indices = np.arange(len(y))
train_idx, valid_idx = train_test_split(
    row_indices, test_size=0.2, random_state=42, stratify=y
)
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

context_cols = ["Type of Travel", "Customer Type", "Class"]
X_context = X.copy()
X_test_context = X_test.copy()
X_context["Travel_Context"] = X_context[context_cols].astype(str).agg(" | ".join, axis=1)
X_test_context["Travel_Context"] = X_test_context[context_cols].astype(str).agg(" | ".join, axis=1)

# Summarize the passenger's service ratings across the 14 survey items.
service_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
]
# Add Inflight service only if this dataset includes that column.
if "Inflight service" in X.columns:
    service_cols.append("Inflight service")

X_context_summary = X_context.copy()
X_test_context_summary = X_test_context.copy()
for features in (X_context_summary, X_test_context_summary):
    service_ratings = features[service_cols]
    features["Service_Rating_Mean"] = service_ratings.mean(axis=1)
    features["Service_Rating_Low_Count"] = (service_ratings <= 2).sum(axis=1)
    features["Service_Rating_High_Count"] = (service_ratings >= 4).sum(axis=1)
    features["Service_Rating_Std"] = service_ratings.std(axis=1)

feature_sets = {
    "travel_context": (X_context, X_test_context, cat_features + ["Travel_Context"]),
    "travel_context_plus_summaries": (
        X_context_summary,
        X_test_context_summary,
        cat_features + ["Travel_Context"],
    ),
}
models_by_features = {}
scores_by_features = {}

for feature_name, (features, _, categorical_cols) in feature_sets.items():
    print(f"\nTraining CatBoost: {feature_name}")
    candidate = CatBoostClassifier(
        iterations=4000,
        learning_rate=0.03,
        depth=7,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=300,
        early_stopping_rounds=150,
        allow_writing_files=False,
        thread_count=-1,
    )
    candidate.fit(
        features.iloc[train_idx], y_train,
        cat_features=categorical_cols,
        eval_set=(features.iloc[valid_idx], y_valid),
        use_best_model=True,
    )
    candidate_proba = candidate.predict_proba(features.iloc[valid_idx])[:, 1]
    candidate_auc = roc_auc_score(y_valid, candidate_proba)
    models_by_features[feature_name] = candidate
    scores_by_features[feature_name] = candidate_auc
    print(f"{feature_name}: validation ROC AUC={candidate_auc:.6f}; best iteration={candidate.best_iteration_}")

best_feature_name = max(scores_by_features, key=scores_by_features.get)
model = models_by_features[best_feature_name]
X_final, X_test_final, final_cat_features = feature_sets[best_feature_name]
valid_proba = model.predict_proba(X_final.iloc[valid_idx])[:, 1]
best_depth = 7
print(f"\nSelected feature set: {best_feature_name} (AUC={scores_by_features[best_feature_name]:.6f})")



Training CatBoost: travel_context
0:	test: 0.9418300	best: 0.9418300 (0)	total: 482ms	remaining: 32m 6s
300:	test: 0.9560421	best: 0.9560421 (300)	total: 1m 12s	remaining: 14m 47s
600:	test: 0.9570618	best: 0.9570618 (600)	total: 2m 22s	remaining: 13m 27s
900:	test: 0.9573925	best: 0.9573925 (900)	total: 4m 15s	remaining: 14m 37s
1200:	test: 0.9575417	best: 0.9575417 (1200)	total: 5m 26s	remaining: 12m 39s
1500:	test: 0.9576491	best: 0.9576491 (1500)	total: 6m 37s	remaining: 11m 2s
1800:	test: 0.9577132	best: 0.9577132 (1796)	total: 7m 49s	remaining: 9m 33s
2100:	test: 0.9577827	best: 0.9577843 (2091)	total: 9m	remaining: 8m 8s
2400:	test: 0.9578571	best: 0.9578571 (2400)	total: 10m 46s	remaining: 7m 10s
2700:	test: 0.9578992	best: 0.9578995 (2698)	total: 11m 57s	remaining: 5m 45s
3000:	test: 0.9579327	best: 0.9579377 (2984)	total: 13m 11s	remaining: 4m 23s
3300:	test: 0.9579546	best: 0.9579590 (3273)	total: 15m 13s	remaining: 3m 13s
3600:	test: 0.9579874	best: 0.9579874 (3600)	total:

## Optional threshold-based check

 The competition is scored by ROC AUC, so probability ranking is the key result. The next cell is only an optional classification report at threshold 0.5.


In [7]:
valid_pred = (valid_proba >= 0.5).astype(int)
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, valid_pred))


              precision    recall  f1-score   support

           0     0.9239    0.9436    0.9336     77859
           1     0.9273    0.9025    0.9147     62068

    accuracy                         0.9254    139927
   macro avg     0.9256    0.9231    0.9242    139927
weighted avg     0.9254    0.9254    0.9253    139927

Confusion matrix:
 [[73467  4392]
 [ 6050 56018]]


In [8]:
# Refit on all labeled rows using the feature set selected by validation.
best_iterations = model.best_iteration_ + 1 if model.best_iteration_ is not None else None
if best_iterations is None or best_iterations < 1:
    best_iterations = 1500

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.03,
    depth=best_depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)
final_model.fit(X_final, y, cat_features=final_cat_features)
test_proba = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_catboost.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())


0:	total: 892ms	remaining: 59m 26s
200:	total: 1m 19s	remaining: 25m 7s
400:	total: 2m 37s	remaining: 23m 33s
600:	total: 3m 45s	remaining: 21m 17s
800:	total: 5m 11s	remaining: 20m 42s
1000:	total: 6m 44s	remaining: 20m 11s
1200:	total: 8m 24s	remaining: 19m 34s
1400:	total: 10m 16s	remaining: 19m 2s
1600:	total: 12m 21s	remaining: 18m 30s
1800:	total: 14m 46s	remaining: 18m 1s
2000:	total: 17m 20s	remaining: 17m 18s
2200:	total: 19m 59s	remaining: 16m 19s
2400:	total: 22m 46s	remaining: 15m 9s
2600:	total: 25m 17s	remaining: 13m 35s
2800:	total: 27m 47s	remaining: 11m 53s
3000:	total: 30m 22s	remaining: 10m 6s
3200:	total: 32m 51s	remaining: 8m 11s
3400:	total: 35m 21s	remaining: 6m 13s
3600:	total: 37m 52s	remaining: 4m 11s
3800:	total: 40m 24s	remaining: 2m 6s
3998:	total: 42m 53s	remaining: 0us
Saved: C:\Users\Asus\OneDrive\Desktop\Predicting_Airline_Satisfaction\submission_catboost.csv
       id  satisfaction
0  699635      0.222305
1  699636      0.881760
2  699637      0.909788